### “Joint angles were extracted from 2D side-view video using a markerless pose estimation approach. Frames with insufficient landmark visibility were excluded, and short gaps in the angle time series were linearly interpolated to enable robust temporal analysis.”

In [36]:
from Tools.scripts.generate_opcode_h import header

"""
Stable squat-angle extraction from a SIDE-VIEW smartphone video using MediaPipe Pose.

Outputs:
- CSV with per-frame angles + quality signals
- Optional annotated video (skeleton + angles overlay)

Install:
  pip install opencv-python mediapipe numpy pandas

How to Run:
    It's mandatory to change one variable.
    video_directory - set it to the directory path with the video files
    The results are saved in the same path in a subfolder called 'results'.
    Run the notebook.

Last but not least, a list of references for all "not common" libraries I used:
https://chuoling.github.io/mediapipe/solutions/pose.html
https://github.com/AISoltani/MediaPipe_Pose_Face_Hand_Detection
https://docs.opencv.org/3.4/dd/d43/tutorial_py_video_display.html
https://github.com/casiez/OneEuroFilter/tree/main/python
"""

from pathlib import Path
import math
import os
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List
import cv2
import numpy as np
import pandas as pd

# error handling with mediapipe
try:
    import mediapipe as mp
except ImportError as e:
    raise SystemExit("Missing mediapipe. Install with: pip install mediapipe") from e

# -----------------------------
# Geometry helpers (core math)
# goal: compute the angle at point B in triangle A–B–C, in degrees.
# -----------------------------
def angle_abc(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """
    Returns angle ABC in degrees, where points are 2D np arrays [x, y] in pixels.
    """
    # direction vectors (always from mid to outer landmarks)
    ba = a - b                  # b to a
    bc = c - b                  # b to c
    nba = np.linalg.norm(ba)    # length (magnitude) of vector BA
    nbc = np.linalg.norm(bc)    # length (magnitude) of vector BC
    # checking if values are close to 0, to prevent division by zero
    if nba < 1e-6 or nbc < 1e-6:
        return float("nan")
    cosang = float(np.dot(ba, bc) / (nba * nbc))    # cosine of the angle
    cosang = max(-1.0, min(1.0, cosang))            # make sure -1 <= cosang <= 1, to avoid numerical errors
    return math.degrees(math.acos(cosang))

# TODO not included yet
# def vector_angle_to_vertical(p1: np.ndarray, p2: np.ndarray) -> float:
#     """
#     Angle of vector p1->p2 relative to vertical axis (in degrees).
#     0 = perfectly vertical, 90 = horizontal.
#     Useful for torso/shin lean in side view.
#     """
#     v = p2 - p1                 # direction vector
#     nv = np.linalg.norm(v)      # length (magnitude) of v
#     if nv < 1e-6:               # Checks if the vector is almost zero-length
#         return float("nan")
#     # vertical unit (upwards in image is negative y, but angle magnitude is what matters)
#     vert = np.array([0.0, -1.0], dtype=np.float32)
#     cosang = float(np.dot(v / nv, vert))        # dot product with normalized v and vert
#     cosang = max(-1.0, min(1.0, cosang))        # make sure -1 <= cosang <= 1, to avoid numerical errors
#     return math.degrees(math.acos(cosang))

# -----------------------------
# One Euro Filter (stable smoothing)
# -----------------------------
@dataclass
class OneEuroParams:            # container class for parameters
    min_cutoff: float = 1.0     # lower = smoother
    beta: float = 0.02          # higher = more responsive
    d_cutoff: float = 1.0       # derivative cutoff

class OneEuroFilter:
    """
    One Euro Filter for a single scalar time series.
    Reference: Casiez et al. "The 1€ Filter" (commonly used for pose smoothing).
    """
    def __init__(self, params: OneEuroParams, init_value: Optional[float] = None, init_time: Optional[float] = None):
        self.params = params        # config parameters
        self.x_prev = init_value    # previous filtered value
        self.dx_prev = 0.0          # previous filtered derivative
        self.t_prev = init_time     # previous time

    # computes alpha, the exponential smoothing factor
    @staticmethod
    def _alpha(cutoff: float, dt: float) -> float:
        # alpha = 1 / (1 + tau/dt), tau = 1/(2*pi*cutoff)
        if dt <= 0.0:
            return 1.0
        tau = 1.0 / (2.0 * math.pi * cutoff)
        return 1.0 / (1.0 + tau / dt)

    # classic exponential smoothing with lowpass filter
    @staticmethod
    def _lowpass(x: float, x_prev: float, alpha: float) -> float:
        return alpha * x + (1.0 - alpha) * x_prev

    def __call__(self, x: float, t: float) -> float:
        if self.t_prev is None or self.x_prev is None or math.isnan(self.x_prev):
            # initialize filter state with the first observation
            self.t_prev = t
            self.x_prev = x
            self.dx_prev = 0.0
            return x

        # compute time step
        dt = t - self.t_prev
        if dt <= 0.0:           # if timestamps are non-increasing, return last filtered value (avoids unstable math)
            return self.x_prev

        # Derivative of the signal
        dx = (x - self.x_prev) / dt
        a_d = self._alpha(self.params.d_cutoff, dt)
        dx_hat = self._lowpass(dx, self.dx_prev, a_d)

        # Adaptive cutoff
        cutoff = self.params.min_cutoff + self.params.beta * abs(dx_hat)
        a = self._alpha(cutoff, dt)
        x_hat = self._lowpass(x, self.x_prev, a)

        self.x_prev = x_hat
        self.dx_prev = dx_hat
        self.t_prev = t
        return x_hat

# -----------------------------
# MediaPipe landmark extraction
# -----------------------------
def mp_landmarks_to_pixels(landmarks, w: int, h: int) -> Dict[int, Tuple[np.ndarray, float]]:
    """
    Deep-learning output (normalized coordinates)
            ↓
    Biomechanics-ready data (pixel vectors + confidence)
    Returns dict: landmark_index -> (pixel_xy [x,y], visibility)
    """
    out = {}
    for i, lm in enumerate(landmarks.landmark):
        out[i] = (np.array([lm.x * w, lm.y * h], dtype=np.float32), float(getattr(lm, "visibility", 1.0)))
    return out


def pick_visible_side(sample_vis: List[Dict[str, float]]) -> str:
    """
    Decide whether LEFT or RIGHT side is more visible overall.
    Uses median visibility over a short sample.
    """
    left_scores = []
    right_scores = []
    for v in sample_vis:
        left_scores.append(np.mean([v.get("l_hip", 0), v.get("l_knee", 0), v.get("l_ankle", 0), v.get("l_foot", 0)]))
        right_scores.append(np.mean([v.get("r_hip", 0), v.get("r_knee", 0), v.get("r_ankle", 0), v.get("r_foot", 0)]))
    l_med = float(np.median(left_scores)) if left_scores else 0.0
    r_med = float(np.median(right_scores)) if right_scores else 0.0
    return "left" if l_med >= r_med else "right"

# -----------------------------
# Main processing
# -----------------------------
def process_video(
    video_path: str,                # path to input data (.mov file)
    out_csv: str,                   # path to output data (.csv file)
    out_video: Optional[str],       # path to output data (.mov file)
    model_complexity: int = 2,
    detection_confidence: float = 0.7,          # determines how "strictly" the tracking is
    tracking_confidence: float = 0.7,
    visibility_gate: float = 0.6,
    oneeuro: OneEuroParams = OneEuroParams(min_cutoff=1.0, beta=0.02, d_cutoff=1.0),
    sample_frames_for_side: int = 30,
):
    # to avoid path errors
    if not os.path.exists(video_path):
        raise FileNotFoundError(video_path)

    # to avoid OpenCV errors
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    # webcam/video/or other input configs
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # optional annotated video writer
    writer = None
    if out_video:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")    # fourcc = “Four Character Code” (mp4v)
        writer = cv2.VideoWriter(out_video, fourcc, float(fps), (width, height))

    # prepare mediapipe objects
    mp_pose = mp.solutions.pose                 # pose estimator
    mp_draw = mp.solutions.drawing_utils        # used for rendering skeleton onto frame
    mp_styles = mp.solutions.drawing_styles     # used for rendering skeleton onto frame

    # indices we care about
    L = mp_pose.PoseLandmark
    IDX = {
        "l_shoulder": int(L.LEFT_SHOULDER),
        "l_hip": int(L.LEFT_HIP),
        "l_knee": int(L.LEFT_KNEE),
        "l_ankle": int(L.LEFT_ANKLE),
        "l_heel": int(L.LEFT_HEEL),
        "l_foot": int(L.LEFT_FOOT_INDEX),

        "r_shoulder": int(L.RIGHT_SHOULDER),
        "r_hip": int(L.RIGHT_HIP),
        "r_knee": int(L.RIGHT_KNEE),
        "r_ankle": int(L.RIGHT_ANKLE),
        "r_heel": int(L.RIGHT_HEEL),
        "r_foot": int(L.RIGHT_FOOT_INDEX),
    }

    #--------------------------------------------------------------------------------------------------------
    # The following section could be skipped for the sake of argument, as I have only used left-handed data.
    # first (short) pass to pick a stable visible side
    sample_visibility = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # rewind to the beginning of the video
    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=model_complexity,
        enable_segmentation=False,
        smooth_landmarks=True,                          # MediaPipe’s own internal smoothing
        min_detection_confidence=detection_confidence,
        min_tracking_confidence=tracking_confidence,
    ) as pose:
        for i in range(sample_frames_for_side):
            ok, frame = cap.read()
            if not ok:
                break
            # OpenCV uses BGR format, MediaPipe expects RGB, so we need to convert it
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)
            if res.pose_landmarks:
                pts = mp_landmarks_to_pixels(res.pose_landmarks, width, height)
                v = {k: pts[IDX[k]][1] for k in ["l_hip", "l_knee", "l_ankle", "l_foot", "r_hip", "r_knee", "r_ankle", "r_foot"]}
                sample_visibility.append(v)

    # side selection
    side = pick_visible_side(sample_visibility) if sample_visibility else "left"
    # Reset for full processing
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    #--------------------------------------------------------------------------------------------------------

    # One Euro filters on angles (more stable than raw per-frame)
    f_knee = OneEuroFilter(oneeuro)
    f_ankle = OneEuroFilter(oneeuro)
    f_hip = OneEuroFilter(oneeuro)

    rows = []
    frame_idx = 0

    # second (large) pass to do the pose estimation
    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=model_complexity,
        enable_segmentation=False,
        smooth_landmarks=True,
        min_detection_confidence=detection_confidence,
        min_tracking_confidence=tracking_confidence,
    ) as pose:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            # compute time
            time_in_seconds = frame_idx / float(fps)                # in seconds
            time_in_milliseconds = int(time_in_seconds * 1000)      # in milliseconds

            # pose estimation on this frame
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)

            # default values
            knee_ang = ankle_ang = hip_ang = float("nan")
            gate_ok = False

            # if landmarks were detected, convert landmarks to pixel coords + visibility
            if res.pose_landmarks:
                pts = mp_landmarks_to_pixels(res.pose_landmarks, width, height)

                if side == "left":
                    shoulder, v_sh = pts[IDX["l_shoulder"]]
                    hip, v_hip = pts[IDX["l_hip"]]
                    knee, v_knee = pts[IDX["l_knee"]]
                    ankle, v_ank = pts[IDX["l_ankle"]]
                    #heel, v_heel = pts[IDX["l_heel"]]
                    foot, v_foot = pts[IDX["l_foot"]]
                else:
                    shoulder, v_sh = pts[IDX["r_shoulder"]]
                    hip, v_hip = pts[IDX["r_hip"]]
                    knee, v_knee = pts[IDX["r_knee"]]
                    ankle, v_ank = pts[IDX["r_ankle"]]
                    #heel, v_heel = pts[IDX["r_heel"]]
                    foot, v_foot = pts[IDX["r_foot"]]

                # Visibility gating for stability, prevents using garbage landmarks (occlusion, blur, misdetection)
                visibility_mean = float(np.mean([v_hip, v_knee, v_ank, v_foot]))
                gate_ok = visibility_mean >= visibility_gate

                if gate_ok:
                    # Core angles
                    knee_ang = angle_abc(hip, knee, ankle)          # hip-knee-ankle
                    ankle_ang = angle_abc(knee, ankle, foot)        # knee-ankle-foot (dorsi/plantar proxy)
                    hip_ang = angle_abc(shoulder, hip, knee)  # shoulder-hip-knee (hip+trunk proxy)

                    # One Euro smoothing
                    knee_ang = f_knee(knee_ang, time_in_seconds)
                    ankle_ang = f_ankle(ankle_ang, time_in_seconds)
                    hip_ang = f_hip(hip_ang, time_in_seconds)

                # Draw overlay (optional)
                if writer is not None:
                    mp_draw.draw_landmarks(
                        frame,
                        res.pose_landmarks,
                        mp_pose.POSE_CONNECTIONS,
                        landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style(),
                    )

            # csv output data
            rows.append(
                {
                    "time_ms": time_in_milliseconds,
                    "hip_deg": hip_ang,
                    "knee_deg": knee_ang,
                    "ankle_deg": ankle_ang,
                    "frame": frame_idx,
                    "side_used": side,
                    "gate_ok": int(gate_ok),
                }
            )

            # video annotation
            if writer is not None:
                # Text overlay
                txt = f"side={side} gate={int(gate_ok)}  knee={knee_ang:6.1f}  ankle={ankle_ang:6.1f}  hip*={hip_ang:6.1f}"
                cv2.putText(frame, txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (20, 20, 20), 4, cv2.LINE_AA)
                cv2.putText(frame, txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (245, 245, 245), 1, cv2.LINE_AA)
                writer.write(frame)

            frame_idx += 1

    # clean up video handles
    cap.release()
    if writer is not None:
        writer.release()

    df = pd.DataFrame(rows)

    # optional: fill short gaps (frames failing gate) by interpolation for downstream rep segmentation
    # keep original columns; add *_interpolate variants
    for col in ["knee_deg", "ankle_deg", "hip_deg"]:
        s = df[col].astype(float)
        df[col + "_interpolate"] = s.interpolate(limit=10, limit_direction="both")

    # if everything worked correctly, the following information will be output in the Jupyter notebook.
    df.to_csv(out_csv, index=False, float_format="%.4f")
    print(f"Done.\n- CSV: {out_csv}\n- Annotated video: {out_video if out_video else '(not written)'}")
    print(f"Side chosen: {side} | Frames processed: {len(df)} | FPS: {fps:.2f} | Resolution: {width}x{height}")


'''
The following last section is where the video_directory path need to be set
'''
# video_directory = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\Datenaufnahme"
# output_dir = str(Path(video_directory).parent) + "\\results\\"
video_directory = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\DatenaufnahmeTest"
output_dir = str(Path(video_directory).parent) + "\\resultsTest\\"
model_complexity = 2

for file in os.listdir(video_directory):
    video_path = video_directory + "\\" + file
    if not video_path.endswith(".mp4"):
        continue
    out_csv = output_dir + "\\" + Path(video_path).stem + "_" + str(model_complexity) + ".csv"
    out_video = output_dir + "\\" + Path(video_path).stem + "_" + str(model_complexity) + ".mp4"

    params = OneEuroParams(
        min_cutoff=1.0,   # smoother if lower
        beta=0.02,        # responsiveness
        d_cutoff=1.0
    )

    process_video(
        video_path=video_path,
        out_csv=out_csv,
        out_video=out_video,
        model_complexity=model_complexity,   # highest accuracy
        detection_confidence=0.7,
        tracking_confidence=0.7,
        visibility_gate=0.6,
        oneeuro=params,
    )

Done.
- CSV: C:\Users\mauri\Desktop\Uni\WS_25_26\Bachelor\PoseEstimation\resources\resultsTest\\IMG_0017_2.csv
- Annotated video: C:\Users\mauri\Desktop\Uni\WS_25_26\Bachelor\PoseEstimation\resources\resultsTest\\IMG_0017_2.mp4
Side chosen: left | Frames processed: 447 | FPS: 29.97 | Resolution: 1080x1920


In [40]:
# -----------------------------
# Comparison with reference
# -----------------------------
# -----------------------------
# File paths
# -----------------------------
pre = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\\resultsTest\IMG_0017_2.csv"
ref = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\\realData\\1_participant_m_b_modified.csv"
out = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\\resultsTest\differences"
# Output files
diff_output = "differences.csv"
mae_output = "mae_results.csv"

# -----------------------------
# Load CSV files
# -----------------------------
def calculate_differences(reference, prediction, output):
    """
    Compares the first 4 columns of two CSV files row-by-row and computes:
      - differences (reference - prediction)
      - MAE per column
    Writes:
      - <output_prefix>_differences.csv
      - <output_prefix>_mae_results.csv
    """

    ref = pd.read_csv(reference)
    pred = pd.read_csv(prediction)

    # First 4 columns only (by position)
    ref = ref.iloc[:, :4]
    pred = pred.iloc[:, :4]

    if ref.shape != pred.shape:
        print("Shapes:", ref.shape, pred.shape)
        raise ValueError(f"CSV shapes do not match: {ref.shape} vs {pred.shape}")

    ref = ref.apply(pd.to_numeric, errors="raise")
    pred = pred.apply(pd.to_numeric, errors="raise")
    diff_arr = ref.to_numpy() - pred.to_numpy()
    diff_df = pd.DataFrame(diff_arr, columns=ref.columns)

    # MAE per column
    mae_vals = np.mean(np.abs(diff_arr), axis=0)
    mae_df = pd.DataFrame({"Column": ref.columns, "MAE": mae_vals})

    # Save outputs
    output_prefix = str(Path(output))
    diff_path = f"{output_prefix}\differences.csv"
    mae_path = f"{output_prefix}\mae_results.csv"

    diff_df.to_csv(diff_path, index=False)
    mae_df.to_csv(mae_path, index=False)

    print("\nMAE per column:")
    print(mae_df)

    return diff_df, mae_df, diff_path, mae_path

calculate_differences(ref, pre, out)


MAE per column:
    Column        MAE
0  time_ms  26.228188
1      hip   4.267680
2     knee   4.074156
3    ankle   1.928333


(     time_ms      hip    knee   ankle
 0       25.0  -3.3491 -1.7553  1.2131
 1       25.0  -5.1429 -3.3138  0.8734
 2       26.0  -6.9335 -4.9207  0.5664
 3       25.0  -8.7134 -6.6066  0.1271
 4       25.0 -10.1490 -8.1440 -0.4195
 ..       ...      ...     ...     ...
 442     27.0  -1.1007  3.1599  2.4307
 443     27.0  -1.6264  1.7398  2.8903
 444     27.0  -1.4623  1.4662  2.5903
 445     27.0  -1.0808  1.5139  1.8670
 446     27.0  -0.7042  2.0352  1.0543
 
 [447 rows x 4 columns],
     Column        MAE
 0  time_ms  26.228188
 1      hip   4.267680
 2     knee   4.074156
 3    ankle   1.928333,
 'C:\\Users\\mauri\\Desktop\\Uni\\WS_25_26\\Bachelor\\PoseEstimation\\resources\\resultsTest\\differences\\differences.csv',
 'C:\\Users\\mauri\\Desktop\\Uni\\WS_25_26\\Bachelor\\PoseEstimation\\resources\\resultsTest\\differences\\mae_results.csv')

In [25]:
def convert_european_csv_stream(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            line = line.replace('"', "")
            line = line.replace(",", ".")
            line = line.replace(";", ",")
            fout.write(line)

i = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\\realData\\1_participant_m_b_IMG_0017_improved.csv"
o = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\\realData\\1_participant_m_b_modified.csv"

convert_european_csv_stream(i,o)